# NISAR GSLC GIF Generator

Create animated GIFs of NISAR GSLC (Geocoded SLC) amplitude time series
using `opera_utils` for search, download/streaming, and subsetting.

Adapted from [NISAR_gif-v1.ipynb](https://github.com/piyushrpt/dl-notebooks/blob/main/gifs/NISAR_gif-v1.ipynb)
(EarthDaily/Descartes Labs) to use `opera_utils.nisar` for data access.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path

import h5py
import matplotlib.animation as anim
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import PillowWriter

from opera_utils.nisar import GslcProduct, search

## 1. Utilities

In [ ]:
def multilook(arr: np.ndarray, looks_row: int, looks_col: int) -> np.ndarray:
    """Apply spatial multilooking (averaging) to reduce noise."""
    nrows, ncols = arr.shape
    new_rows = nrows // looks_row
    new_cols = ncols // looks_col
    arr = arr[: new_rows * looks_row, : new_cols * looks_col]
    return arr.reshape(new_rows, looks_row, new_cols, looks_col).mean(axis=(1, 3))


class AnimatedGif:
    """Matplotlib-based animated GIF helper."""

    def __init__(self, size=(640, 480)):
        self.fig = plt.figure()
        self.fig.set_size_inches(size[0] / 100, size[1] / 100)
        ax = self.fig.add_axes([0, 0, 1, 1], frameon=False, aspect=1)
        ax.set_xticks([])
        ax.set_yticks([])
        self.images = []

    def add(self, image, label="", clim=(0, 1), cmap="gray", **fontargs):
        plt_im = plt.imshow(
            image, cmap=cmap, vmin=clim[0], vmax=clim[1], animated=True
        )
        plt_txt = plt.text(10, 30, label, **fontargs)
        self.images.append([plt_im, plt_txt])

    def save(self, filename, fps=3):
        animation = anim.ArtistAnimation(self.fig, self.images)
        animation.save(filename, writer=PillowWriter(fps=fps))

## 2. Search for GSLC products

Use `opera_utils.nisar.search()` to query NASA's CMR for GSLC products
intersecting a bounding box. You can also filter by orbit parameters
and time range.

In [ ]:
# Area of interest (west, south, east, north)
# This bbox is in Ethiopia, covered by NISAR orbit 172, frame 8, ascending
BBOX = (40.62, 13.56, 40.72, 13.64)

products = search(
    bbox=BBOX,
    start_datetime=datetime(2025, 9, 1),
    end_datetime=datetime(2026, 3, 1),
)

print(f"Found {len(products)} GSLC products:")
for p in products:
    print(
        f"  {p.start_datetime.date()}"
        f"  orbit={p.relative_orbit_number}"
        f"  frame={p.track_frame_number}"
        f"  dir={p.orbit_direction}"
        f"  {Path(p.filename).name}"
    )

In [ ]:
# Group products by (orbit, frame, direction) to get distinct time series
from collections import defaultdict

groups: dict[str, list[GslcProduct]] = defaultdict(list)
for p in products:
    key = f"{p.relative_orbit_number:03d}_{p.track_frame_number:03d}_{p.orbit_direction}"
    groups[key].append(p)

print("Available track/frame groups:")
for key, prods in groups.items():
    print(f"  {key}: {len(prods)} acquisitions")

In [ ]:
# Pick a specific track/frame group, or just use all products if they share
# the same geometry. Edit the key below as needed.
# For a single track, you can also search more specifically:
#   products = search(bbox=BBOX, relative_orbit_number=172, track_frame_number=8,
#                     orbit_direction="A")

TRACK_KEY = list(groups.keys())[0]  # pick the first group
selected = groups[TRACK_KEY]
print(f"Using {TRACK_KEY} with {len(selected)} products")

## 3. Download/subset or stream GSLC data

Two approaches are available:

**Option A – Stream directly** (no local files needed, requires Earthdata credentials):
Uses `opera_utils.nisar.open_h5()` to stream data from HTTPS/S3.

**Option B – Download subsets first** (faster for repeated use):
Uses `opera_utils.nisar.run_download()` to download spatially subsetted files.

In [ ]:
# Option B: Download subsetted files (recommended for GIF creation)
from opera_utils.nisar import run_download

output_dir = Path("./gslc_subsets")
downloaded = run_download(
    bbox=BBOX,
    start_datetime=datetime(2025, 9, 1),
    end_datetime=datetime(2026, 3, 1),
    polarizations=["HH"],
    output_dir=output_dir,
    num_workers=4,
)
downloaded = sorted(downloaded)
print(f"Downloaded {len(downloaded)} subsetted files to {output_dir}")

## 4. Create the GIF

In [ ]:
FREQUENCY = "A"
POLARIZATION = "HH"
# Number of looks for spatial averaging (reduce noise in amplitude)
LOOKS_ROW = 5
LOOKS_COL = 5


def load_amplitude(filepath: Path) -> tuple[np.ndarray, str]:
    """Load multilooked amplitude from a GSLC file and extract its date label."""
    product = GslcProduct.from_filename(filepath)
    date_label = product.start_datetime.strftime("%Y-%m-%d")

    with h5py.File(filepath, "r") as f:
        dset_path = product.get_dataset_path(FREQUENCY, POLARIZATION)
        slc = f[dset_path][:]

    amp = np.abs(slc).astype(np.float32)
    amp = multilook(amp, LOOKS_ROW, LOOKS_COL)
    return amp, date_label

In [ ]:
# Load all frames
frames = []
for filepath in downloaded:
    amp, label = load_amplitude(filepath)
    frames.append((amp, label))
    print(f"  {label}: shape={amp.shape}, min={amp.min():.3f}, max={amp.max():.3f}")

# Compute a common color scale from the stack (use 98th percentile as vmax)
all_amps = np.concatenate([f[0].ravel() for f in frames])
vmax = float(np.nanpercentile(all_amps, 98))
print(f"\nUsing vmax={vmax:.3f} (98th percentile)")

In [ ]:
# Build the animated GIF
first_shape = frames[0][0].shape
animated_gif = AnimatedGif(size=(first_shape[1], first_shape[0]))

fontargs = {"fontsize": "large", "color": "red", "backgroundcolor": "white"}
for amp, label in frames:
    animated_gif.add(amp, label=label, clim=(0, vmax), cmap="gray", **fontargs)

outname = "nisar_gslc_amplitude.gif"
animated_gif.save(outname, fps=3)
print(f"Saved {outname}")

In [ ]:
# Display the GIF inline (Jupyter)
from IPython.display import Image, display

display(Image(filename=outname))

## 5. Alternative: Stream directly without downloading

If you don't want to store local files, you can stream each frame
directly from the archive using `open_h5()`.

In [ ]:
# from opera_utils.nisar import open_h5
#
# frames = []
# for product in selected:
#     date_label = product.start_datetime.strftime("%Y-%m-%d")
#     with open_h5(str(product.filename)) as hf:
#         dset_path = product.get_dataset_path(FREQUENCY, POLARIZATION)
#
#         # Use lonlat_to_rowcol to get the spatial subset matching our bbox
#         west, south, east, north = BBOX
#         row0, col0 = product.lonlat_to_rowcol(west, north, FREQUENCY)
#         row1, col1 = product.lonlat_to_rowcol(east, south, FREQUENCY)
#         r_start, r_end = min(row0, row1), max(row0, row1)
#         c_start, c_end = min(col0, col1), max(col0, col1)
#
#         slc = hf[dset_path][r_start:r_end, c_start:c_end]
#
#     amp = multilook(np.abs(slc).astype(np.float32), LOOKS_ROW, LOOKS_COL)
#     frames.append((amp, date_label))
#     print(f"  {date_label}: {amp.shape}")